* 方法参考自 The Barra China Equity Model (CNE5)'s 文档

* 请在环境变量中设置`DB_URI`指向数据库

In [ ]:
import asyncio
import datetime as dt


In [ ]:
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc


In [ ]:
rng = np.random.default_rng(42)
n_dates = 6


In [ ]:
n_assets = 12
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])


In [ ]:
n_rows = n_dates * n_assets
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})


In [ ]:
from alphamind.data.neutralize import neutralize

risk = np.column_stack([
    panel["BETA"].to_numpy(zero_copy_only=False),
    panel["SIZE"].to_numpy(zero_copy_only=False),
])


In [ ]:
signal = panel["factor"].to_numpy(zero_copy_only=False).reshape(-1, 1)
residual = neutralize(risk, signal)
assert np.allclose(risk.T @ residual, 0.0, atol=1e-8)


In [ ]:
result = pa.table({
    "trade_date": panel["trade_date"],
    "code": panel["code"],
    "residual": residual[:, 0],
})
result
